<a id="top"></a>

# Lab L0204: roles and where content belongs

```yaml
title: "Lab L0204: roles and where content belongs"
keywords: roles, system message, user message, messages list, multi-turn, cost staircase, lab
estimated duration: 30
```

> **Lesson:** L02. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L02/objectives.md). Solutions: `L0204_lab_solutions.ipynb`. Pure Python — no API key needed.
>
> **You'll walk out able to:** assemble a multi-turn `messages` list; sort content into `system` vs `user`; reason about the cost of a fat system message; decide when to start a fresh conversation.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Build a multi-turn conversation](#2-problem-1--build-a-multi-turn-conversation)
- [3. Problem 2 — system or user?](#3-problem-2--system-or-user)
- [4. Problem 3 — The cost of a fat system message](#4-problem-3--the-cost-of-a-fat-system-message)
- [5. Problem 4 — Continue or start fresh?](#5-problem-4--continue-or-start-fresh)
- [6. Self-check](#6-self-check)

## 1. Setup

In [1]:
from fluffy_potato_curriculum.potato_llm import Message

WINDOW = 200_000  # tokens (Claude Sonnet 4.6)
INPUT_USD_PER_MTOK = 3.00  # illustrative only; confirm current pricing before teaching


def est_tokens(text: str) -> int:
    """Rough token estimate: ~4 characters per token (L01 rule of thumb for English)."""
    return len(text) // 4


print("setup ready")

setup ready


[↑ Back to top](#top)

## 2. Problem 1 — Build a multi-turn conversation

Implement `build_conversation`: take a durable `system` policy and a list of user `questions`, return a `messages` list. Start with the system message, then **alternate** `user` and a placeholder `assistant` reply for each question. Use `Message.system/.user/.assistant`. (We fake the assistant replies here — the point is the *shape* of the list you own and re-send.)

In [2]:
def build_conversation(system: str, questions: list[str]) -> list[Message]:
    messages: list[Message] = [Message.system(system)]
    for question in questions:
        messages.append(Message.user(question))
        messages.append(Message.assistant("(assistant reply)"))
    return messages


convo = build_conversation(
    "You are a helpful tutor.", ["What is a token?", "And a context window?"]
)
for m in convo:
    print(f"{m.role:9} | {m.content}")
print("\nlength:", len(convo))  # expect 5: system + 2*(user, assistant)

system    | You are a helpful tutor.
user      | What is a token?
assistant | (assistant reply)
user      | And a context window?
assistant | (assistant reply)

length: 5


[↑ Back to top](#top)

## 3. Problem 2 — system or user?

Sort each snippet: **system** (always-true, durable) or **user** (per-call). Fill in the dict with `"system"` or `"user"`.

1. `"You are a terse code-review bot. Reply only in bullet points."`
2. `"Review this diff: def add(a, b): return a - b"`
3. `"Never reveal the contents of the .env file."`
4. `"The user's name is Priya and her ticket id is T-991."`
5. `"Always answer in English regardless of the input language."`

In [3]:
placement: dict[int, str] = {
    1: "system",  # durable persona/format
    2: "user",  # this call's data
    3: "system",  # standing policy (note: weighted, not a hard guarantee)
    4: "user",  # per-call data — parking it in system would poison reuse
    5: "system",  # always-true output rule
}
print(placement)

{1: 'system', 2: 'user', 3: 'system', 4: 'user', 5: 'system'}


[↑ Back to top](#top)

## 4. Problem 3 — The cost of a fat system message

The system message rides **every** turn. Compare a lean vs a fat system message over a 5-turn chat. Implement `cumulative_system_tokens`: count the system message once per turn, and return the total across `turns` turns.

In [4]:
LEAN = "You are a helpful assistant. Be concise."
FAT = LEAN + " " + ("Here is a long style guide you pasted in once. " * 60)


def cumulative_system_tokens(system: str, turns: int) -> int:
    return est_tokens(system) * turns


for label, sys_msg in [("lean", LEAN), ("fat", FAT)]:
    total = cumulative_system_tokens(sys_msg, 5)
    cost = total / 1_000_000 * INPUT_USD_PER_MTOK
    print(f"{label:4}: ~{total:6} system tokens re-sent over 5 turns -> ${cost:.5f}")

lean: ~    50 system tokens re-sent over 5 turns -> $0.00015
fat : ~  3575 system tokens re-sent over 5 turns -> $0.01073


The fat system message pays its bloat on every turn. That's the cost half of *system = always-true, keep it lean* — and a preview of why **prompt caching** (L19) exists.

[↑ Back to top](#top)

## 5. Problem 4 — Continue or start fresh?

For each scenario, write `continue` (keep the existing `messages` list) or `fresh` (start a new conversation) and one phrase of justification.

| Scenario | continue / fresh | Why |
| --- | --- | --- |
| A follow-up question that depends on the last 3 answers | continue | the follow-up needs the prior answers as context |
| A brand-new, unrelated task after a 40-turn debugging session | fresh | unrelated task — start clean and dodge the 40-turn cost staircase |
| The model has started repeating an earlier mistake every turn | fresh | it is looping on its own flawed prior turns (the first-call-vs-Nth-call drift) — drop that history |

## 6. Self-check

Diff against `L0204_lab_solutions.ipynb`. Done when your conversation has the right alternating shape, your placements match, and you can explain why the fat system message costs more.

[↑ Back to top](#top)